In [1]:
#prep relevent packages 
from pathlib import Path
import pandas as pd
import re
import pdfplumber 

In [2]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'

# import cfp pdfs data
general_plan = raw_fldr / 'cfp_data' / 'CFP-5-yr-PIF-County-FY26-All-261020.pdf'


In [6]:
#check dataframe to see if it can be read  
general_plan.head()

,county,growth_2026,policies_2026,growth_2025,policies_2025,growth_2024,policies_2024,growth_2023,policies_2023,growth_2022,policies_2022
0,El Dorado,0,28134,18,28167,28,23921,21,18715,10,15454
1,Nevada,2,23911,18,23438,17,19795,27,16883,16,13269
2,Placer,0,19053,21,18996,28,15674,20,12220,11,10201
3,Riverside,20,68340,70,57026,95,33472,43,17207,29,12054
4,San Bernardino,10,71352,28,65132,23,50761,14,41124,16,36131


In [ ]:
rows = []  # reset each run so reruns don't duplicate rows

#help provided with some AI, particularly with county row check to get unified, scraped dataframe
with pdfplumber.open(general_plan) as pdf:
    for page in pdf.pages:
        for line in page.extract_text().split('\n'):
            # clean: drop $ (keep % — it marks where each growth value ends)
            line = line.replace('$', '')
            # county name = leading words, up to the first percent
            m = re.match(r'([A-Za-z .]+?)\s+(-?\d+%.*)$', line)
            if m is None or m.group(1).strip() == 'Total':
                continue
            county, rest = m.group(1).strip(), m.group(2)
            # each pair = growth% + count (count may contain stray spaces, ends before next growth%)
            pairs = re.findall(r'(-?\d+)%\s+([\d ,]+?)(?=\s+-?\d+%|\s*$)', rest)
            if len(pairs) == 5:
                values = []
                for growth, count in pairs:
                    values += [growth, count.replace(' ', '').replace(',', '')]
                rows.append([county] + values)

cols = ['county',
        'growth_2026', 'policies_2026',
        'growth_2025', 'policies_2025',
        'growth_2024', 'policies_2024',
        'growth_2023', 'policies_2023',
        'growth_2022', 'policies_2022']
plan_df = pd.DataFrame(rows, columns=cols)
plan_df[cols[1:]] = plan_df[cols[1:]].apply(pd.to_numeric)


58
686376


In [7]:
plan_df.head()

,county,growth_2026,policies_2026,growth_2025,policies_2025,growth_2024,policies_2024,growth_2023,policies_2023,growth_2022,policies_2022
0,Alameda,9,12791,73,11694,92,6774,19,3519,12,2956
1,Alpine,-49,198,13,392,16,346,13,297,21,263
2,Amador,0,6175,13,6186,16,5495,17,4742,11,4041
3,Butte,-2,9161,15,9392,39,8165,58,5878,14,3722
4,Calaveras,0,10609,11,10572,12,9548,22,8520,15,6983


In [10]:
#converting df into appropriate variable types 

plan_df_clean = (
    plan_df
    .set_index('county') #set county aside so it isn't coerced to NaN
    .apply(pd.to_numeric, errors='coerce') #converting strings to numbers, '-' becomes NaN
    .reset_index()
    .assign( #converting into percentages 
        growth_2026 = lambda d: d['growth_2026'] / 100,
        growth_2025 = lambda d: d['growth_2025'] / 100,
        growth_2024 = lambda d: d['growth_2024'] / 100,
        growth_2023 = lambda d: d['growth_2023'] / 100,
        growth_2022 = lambda d: d['growth_2022'] / 100
    )
    .astype({
        'county': 'string',
        'growth_2026': 'float64',
        'policies_2026': 'Int64',
        'growth_2025': 'float64',
        'policies_2025': 'Int64',
        'growth_2024': 'float64',
        'policies_2024': 'Int64',
        'growth_2023': 'float64',
        'policies_2023': 'Int64',
        'growth_2022': 'float64',
        'policies_2022': 'Int64'
        })
)


In [11]:
plan_df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   county         58 non-null     string 
 1   growth_2026    58 non-null     float64
 2   policies_2026  58 non-null     Int64  
 3   growth_2025    58 non-null     float64
 4   policies_2025  58 non-null     Int64  
 5   growth_2024    58 non-null     float64
 6   policies_2024  58 non-null     Int64  
 7   growth_2023    58 non-null     float64
 8   policies_2023  58 non-null     Int64  
 9   growth_2022    58 non-null     float64
 10  policies_2022  58 non-null     Int64  
dtypes: Int64(5), float64(5), string(1)
memory usage: 5.8 KB


In [12]:
plan_df_clean.head()

,county,growth_2026,policies_2026,growth_2025,policies_2025,growth_2024,policies_2024,growth_2023,policies_2023,growth_2022,policies_2022
0,Alameda,0.09,12791,0.73,11694,0.92,6774,0.19,3519,0.12,2956
1,Alpine,-0.49,198,0.13,392,0.16,346,0.13,297,0.21,263
2,Amador,0.00,6175,0.13,6186,0.16,5495,0.17,4742,0.11,4041
3,Butte,-0.02,9161,0.15,9392,0.39,8165,0.58,5878,0.14,3722
4,Calaveras,0.00,10609,0.11,10572,0.12,9548,0.22,8520,0.15,6983


In [14]:
#check to see if numbers from pull are aligned with pdf 
# Plans from pdf
# 2026 plans = 686,376
# 2025 plans = 642,010
# 2024 plans = 463,158
total_2026_policies = plan_df_clean['policies_2026'].sum()
total_2025_policies = plan_df_clean['policies_2025'].sum()
total_2024_policies = plan_df_clean['policies_2024'].sum()

print('2026 plans:',total_2026_policies)
print('2025 plans:',total_2025_policies)
print('2024 plans:',total_2024_policies)



2026 plans: 686376
2025 plans: 642010
2024 plans: 463158


In [22]:
#Enables a quick review of the pdf to ensure specific values in plan_df_clean align with the pdf 
county_check = 'Alameda'
year = 2025
policy_check=plan_df_clean.loc[plan_df_clean['county']==county_check,f'policies_{year}'].iloc[0]

print('County',county_check)
print('Year:',year)
print('Number of policies:',policy_check)



County Alameda
Year: 2025
Number of policies: 11694


In [23]:
#converting to processed folder csv
plan_df_clean.to_csv(processed_fldr / 'cfp_general_plans_cleaned.csv')
